# תרגיל בית מס׳ 4 — מערכות בבינה מלאכותית
## מערכת סוכנים (Agentic AI) במחברת Jupyter

במחברת זו נבנה מערכת סוכנים בסיסית שמקבלת בקשת משתמש, מסווגת את הכוונה שלו, בוחרת כלי מתאים, מפעילה סוכן מתאים, שומרת זיכרון קצר‑טווח ומפעילה מנגנון גיבוי כשרמת הביטחון נמוכה או כשתשובת הסוכן לא איכותית מספיק.

> **הערה:** במימוש זה אנו משתמשים ב־OpenAI API (מודל `gpt-4o-mini`) במקום Ollama, כפי שהותאם לדרישות. שם הסוכן נשמר כ־`LocalLLMAgent` לצורך תאימות עם בדיקות הוולידציה.

## חלק א׳ — הכנת סביבת העבודה במחברת

נתקין ונייבא את כל התלויות הנדרשות. מפתח ה־API של OpenAI נטען מקובץ `.env` באמצעות `python-dotenv`.
אם המפתח חסר או שהקריאה ל־API נכשלת, המערכת תיפול חזרה לתשובה מקומית חלופית.

In [16]:
# Cell 1 - imports and environment setup
import json
import re
import os
from dataclasses import dataclass
from typing import Dict, Any, Optional, List

try:
    import requests
except ImportError:
    requests = None

from dotenv import load_dotenv
load_dotenv()

# OpenAI setup
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_client = None
if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ OpenAI API key loaded successfully.")
else:
    print("⚠️ OPENAI_API_KEY not found in .env — will use local fallback answers.")


✅ OpenAI API key loaded successfully.


## חלק ב׳ — הגדרת מבנה משותף לסוכנים

כל סוכן יחזיר אובייקט `AgentResult` אחיד הכולל את שם הסוכן, התוצאה, רמת הביטחון ופרטי מטא‑דאטה. מבנה אחיד מקל על סוכן התזמור לחבר בין הרכיבים.

In [17]:
# Cell 2 - shared result object
@dataclass
class AgentResult:
    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None


## חלק ג׳ — סוכן ניתוב לסיווג כוונות

סוכן הניתוב מקבל את הודעת המשתמש ומחזיר כוונה (`intent`) ורמת ביטחון (`confidence`). 
המימוש כאן הוא דטרמיניסטי — מבוסס על מילות מפתח — כדי לאפשר בדיקות יציבות וחזרתיות.

| קלט לדוגמה | תוצאה צפויה |
|---|---|
| Tell me a short joke | `generalChat`, 0.82 |
| Analyze this review: ... | `analyzeReview`, 0.93 |
| Summarize the following text | `summarizeText`, 0.90 |
| ? | `unknown`, 0.40 |

In [18]:
# Cell 3 - router agent
INTENTS = ["generalChat", "analyzeReview", "summarizeText", "unknown"]

def router_agent(user_text: str) -> AgentResult:
    text = user_text.lower()
    if any(word in text for word in ["review", "sentiment"]):
        intent, conf = "analyzeReview", 0.93
    elif any(word in text for word in ["summarize", "summary"]):
        intent, conf = "summarizeText", 0.90
    elif len(text.strip()) < 4:
        intent, conf = "unknown", 0.40
    else:
        intent, conf = "generalChat", 0.82
    return AgentResult("RouterAgent", {"intent": intent, "confidence": conf}, conf)


### בדיקות סוכן הניתוב

In [19]:
# Cell 4 - router tests
router_tests = [
    "Tell me a short joke",
    "Analyze this review: The product is amazing",
    "Summarize the following text",
    "?"
]

for text in router_tests:
    result = router_agent(text)
    print(text, "=>", result.output)


Tell me a short joke => {'intent': 'generalChat', 'confidence': 0.82}
Analyze this review: The product is amazing => {'intent': 'analyzeReview', 'confidence': 0.93}
Summarize the following text => {'intent': 'summarizeText', 'confidence': 0.9}
? => {'intent': 'unknown', 'confidence': 0.4}


## חלק ד׳ — סוכן שפה מקומי (OpenAI API)

סוכן השפה אחראי למשימות כלליות — ניסוח, שיחה פשוטה וסיכום בסיסי.

במימוש זה אנו קוראים ל‑**OpenAI API** עם מודל `gpt-4o-mini` (חסכוני ומהיר).
אם המפתח חסר או שהקריאה נכשלת, המערכת תיפול אוטומטית לתשובה מקומית חלופית.

> **שם הסוכן נשמר כ־`LocalLLMAgent`** לצורך תאימות עם בדיקות הוולידציה.

In [20]:
# Cell 5 - OpenAI-based agent with safe fallback
OPENAI_MODEL = "gpt-4o-mini"

def call_openai(prompt: str) -> Optional[str]:
    """Call OpenAI API. Returns None on any failure."""
    if openai_client is None:
        return None
    try:
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Keep your answers concise."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=200,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ OpenAI API call failed: {e}")
        return None

def local_llm_agent(prompt: str) -> AgentResult:
    answer = call_openai(prompt)
    if not answer:
        answer = "Local fallback answer: I received the request and will answer briefly."
    return AgentResult("LocalLLMAgent", answer, 0.80)


### בדיקת סוכן השפה

In [21]:
# Cell 6 - local agent test
result = local_llm_agent("Tell me a short joke about AI agents")
print("Agent name:", result.agent_name)
print("Output:", result.output)


Agent name: LocalLLMAgent
Output: Why did the AI agent break up with its partner? 

It couldn't find the right algorithm for love!


## חלק ה׳ — סוכן כלי ייעודי לניתוח סנטימנט

סוכן זה מנתח את הסנטימנט של טקסט נתון. אם חבילת `transformers` מותקנת — נשתמש במודל מקצועי לניתוח סנטימנט.
אחרת, נשתמש בגישה מבוססת כללים (rule-based) עם מילות מפתח.

| קלט | תוצאה צפויה |
|---|---|
| The product is amazing and I love it | POSITIVE |
| This is terrible and very disappointing | NEGATIVE |
| The product arrived yesterday | NEUTRAL |

In [22]:
# Cell 7 - sentiment agent with optional transformers
try:
    from transformers import pipeline
    sentiment_pipeline = pipeline("sentiment-analysis")
except Exception:
    sentiment_pipeline = None

def rule_based_sentiment(text: str) -> Dict[str, Any]:
    positive = ["good", "great", "amazing", "excellent", "love"]
    negative = ["bad", "terrible", "awful", "hate", "poor"]
    lower = text.lower()
    if any(w in lower for w in positive):
        return {"sentiment": "POSITIVE", "confidence": 0.85}
    if any(w in lower for w in negative):
        return {"sentiment": "NEGATIVE", "confidence": 0.85}
    return {"sentiment": "NEUTRAL", "confidence": 0.60}

def sentiment_agent(text: str) -> AgentResult:
    if sentiment_pipeline:
        result = sentiment_pipeline(text)[0]
        output = {
            "sentiment": result["label"],
            "confidence": round(float(result["score"]), 3)
        }
    else:
        output = rule_based_sentiment(text)
    return AgentResult("SentimentAnalysisAgent", output, output["confidence"])


### בדיקות סוכן הסנטימנט

In [23]:
# Cell 8 - sentiment tests
samples = [
    "The product is amazing and I love it",
    "This is terrible and very disappointing",
    "The product arrived yesterday"
]

for sample in samples:
    print(sample, "=>", sentiment_agent(sample).output)


The product is amazing and I love it => {'sentiment': 'POSITIVE', 'confidence': 0.85}
This is terrible and very disappointing => {'sentiment': 'NEGATIVE', 'confidence': 0.85}
The product arrived yesterday => {'sentiment': 'NEUTRAL', 'confidence': 0.6}


## חלק ו׳ — בחירת כלים

הסוכן אינו רק מייצר טקסט — עליו **לבחור כלי מתאים** לפי הכוונה ורמת הביטחון.
זהו ההבדל המרכזי בין קריאה רגילה למודל לבין מערכת סוכנים אמיתית.

| כוונה | ביטחון | כלי צפוי |
|---|---|---|
| generalChat | 0.82 | local_llm |
| analyzeReview | 0.93 | sentiment_analyzer |
| summarizeText | 0.90 | local_llm |
| unknown | 0.40 | cloud_fallback |

In [24]:
# Cell 9 - tool selection
TOOLS = {
    "local_llm": "general chat and simple summarization",
    "sentiment_analyzer": "sentiment analysis for reviews",
    "cloud_fallback": "fallback when confidence is low"
}

def select_tool(intent: str, confidence: float) -> str:
    if confidence < 0.80:
        return "cloud_fallback"
    if intent == "analyzeReview":
        return "sentiment_analyzer"
    if intent in ["generalChat", "summarizeText"]:
        return "local_llm"
    return "cloud_fallback"


### בדיקות בחירת כלים

In [25]:
# Cell 10 - tool selection tests
cases = [
    ("generalChat", 0.82),
    ("analyzeReview", 0.93),
    ("summarizeText", 0.90),
    ("unknown", 0.40)
]

for intent, confidence in cases:
    print(intent, confidence, "=>", select_tool(intent, confidence))


generalChat 0.82 => local_llm
analyzeReview 0.93 => sentiment_analyzer
summarizeText 0.9 => local_llm
unknown 0.4 => cloud_fallback


## חלק ז׳ — סוכן תזמור מרכזי

סוכן התזמור מחבר את כל המערכת: הוא מקבל את הודעת המשתמש, מפעיל את סוכן הניתוב, בוחר כלי, מפעיל את הסוכן המתאים ומחזיר תשובה סופית.

בנוסף, סוכן התזמור משלב את **סוכן הביקורת** (Critic Agent) — אם ציון האיכות נמוך מ-3, הבקשה מועברת לסוכן הגיבוי.

| תרחיש | סוכן צפוי |
|---|---|
| שיחה כללית | LocalLLMAgent |
| ניתוח ביקורת חיובית | SentimentAnalysisAgent |
| סיכום טקסט | LocalLLMAgent |
| בקשה לא ברורה | CloudFallbackAgent |

In [26]:
# Cell 11 - orchestrator agent (with critic integration)
agent_memory = {
    "last_intent": None,
    "last_tool": None,
    "last_user_message": None,
    "last_result": None
}

def cloud_fallback_agent(user_text: str) -> AgentResult:
    answer = "Fallback answer: the request is unclear or confidence is too low."
    return AgentResult("CloudFallbackAgent", answer, 0.70)

def critic_agent(answer: Any) -> AgentResult:
    """Evaluate the quality of an agent's answer. Score 1-5."""
    text = str(answer)
    if len(text.strip()) < 20:
        score = 2
    elif "example" in text.lower() or "POSITIVE" in text or "NEGATIVE" in text:
        score = 4
    else:
        score = 3
    return AgentResult("CriticAgent", {"quality_score": score}, score / 5)

def orchestrator_agent(user_text: str) -> AgentResult:
    # Step 1: Route the request
    route = router_agent(user_text)
    intent = route.output["intent"]
    confidence = route.output["confidence"]
    tool = select_tool(intent, confidence)

    # Step 2: Execute the appropriate agent
    if tool == "sentiment_analyzer":
        result = sentiment_agent(user_text)
    elif tool == "local_llm":
        result = local_llm_agent(user_text)
    else:
        result = cloud_fallback_agent(user_text)

    # Step 3: Critic evaluation — if quality is below 3, fall back
    critique = critic_agent(result.output)
    if critique.output["quality_score"] < 3:
        result = cloud_fallback_agent(user_text)

    # Step 4: Update memory
    agent_memory.update({
        "last_intent": intent,
        "last_tool": tool,
        "last_user_message": user_text,
        "last_result": result.output
    })
    return result


### בדיקות אינטגרציה

In [27]:
# Cell 12 - integration tests
messages = [
    "Tell me a short joke",
    "Analyze this review: The product is amazing",
    "Summarize the following paragraph",
    "?"
]

for msg in messages:
    result = orchestrator_agent(msg)
    print("USER:", msg)
    print("AGENT:", result.agent_name)
    print("OUTPUT:", result.output)
    print("MEMORY:", agent_memory)
    print("---")


USER: Tell me a short joke
AGENT: LocalLLMAgent
OUTPUT: Why did the scarecrow win an award? 

Because he was outstanding in his field!
MEMORY: {'last_intent': 'generalChat', 'last_tool': 'local_llm', 'last_user_message': 'Tell me a short joke', 'last_result': 'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!'}
---
USER: Analyze this review: The product is amazing
AGENT: SentimentAnalysisAgent
OUTPUT: {'sentiment': 'POSITIVE', 'confidence': 0.85}
MEMORY: {'last_intent': 'analyzeReview', 'last_tool': 'sentiment_analyzer', 'last_user_message': 'Analyze this review: The product is amazing', 'last_result': {'sentiment': 'POSITIVE', 'confidence': 0.85}}
---
USER: Summarize the following paragraph
AGENT: LocalLLMAgent
OUTPUT: Sure! Please provide the paragraph you would like me to summarize.
MEMORY: {'last_intent': 'summarizeText', 'last_tool': 'local_llm', 'last_user_message': 'Summarize the following paragraph', 'last_result': 'Sure! Please provide the paragr

## חלק ח׳ — זיכרון קצר טווח

הזיכרון מאפשר לסוכן להבין בקשות המשך. לדוגמה, לאחר ניתוח ביקורת, המשתמש יכול לשאול "למה?" והמערכת תוכל להתייחס לתוצאה הקודמת.

| שלב | תוצאה צפויה |
|---|---|
| לאחר ניתוח ביקורת | בזיכרון נשמרים הכוונה, הכלי, הודעת המשתמש והתוצאה |
| בקשת המשך: "למה?" | המערכת משתמשת בתוצאה הקודמת כדי להסביר את הסיווג |

In [28]:
# Cell 13 - memory follow-up example
orchestrator_agent("Analyze this review: The product is amazing")
print("Memory after analysis:", agent_memory)
print()

follow_up = "Why?"
if agent_memory["last_intent"] == "analyzeReview":
    print("Explanation: positive words in the review indicate positive sentiment.")
else:
    print("No relevant previous context was found.")


Memory after analysis: {'last_intent': 'analyzeReview', 'last_tool': 'sentiment_analyzer', 'last_user_message': 'Analyze this review: The product is amazing', 'last_result': {'sentiment': 'POSITIVE', 'confidence': 0.85}}

Explanation: positive words in the review indicate positive sentiment.


## חלק ט׳ — סוכן ביקורת (Critic Agent)

סוכן הביקורת מעריך את איכות התשובה של סוכן אחר. הוא מחזיר ציון בין 1 ל-5.
אם הציון נמוך מ-3, סוכן התזמור מפעיל את סוכן הגיבוי במקום (שילוב זה כבר מוטמע בסוכן התזמור למעלה).

| תשובה נבדקת | ציון צפוי |
|---|---|
| תשובה קצרה מאוד (למשל: "כן") | 2 — יש להפעיל גיבוי |
| תשובה מפורטת וברורה | 3 או 4 — ניתן לקבל |
| תשובת סנטימנט מובנית | 4 — בדרך כלל תקינה |

In [29]:
# Cell 14 - critic agent demonstration
# The critic_agent function is already defined in Cell 11.
# Here we demonstrate it separately.

answer = local_llm_agent("Say something short").output
critique = critic_agent(answer)
print("Answer:", answer)
print("Critique:", critique.output)
print()

# Additional examples
test_answers = [
    "Yes",
    "This is a detailed and comprehensive explanation about AI agents and how they work.",
    "POSITIVE with high confidence"
]

for ans in test_answers:
    c = critic_agent(ans)
    print(f"Answer: '{ans}' => Score: {c.output['quality_score']}")


Answer: Sure! What would you like to know?
Critique: {'quality_score': 3}

Answer: 'Yes' => Score: 2
Answer: 'This is a detailed and comprehensive explanation about AI agents and how they work.' => Score: 3
Answer: 'POSITIVE with high confidence' => Score: 4


## חלק י׳ — בדיקות מסכמות

בתא הבא נריץ מספר תרחישים ונוודא שכל רכיבי המערכת עובדים יחד כנדרש.

In [30]:
# Cell 15 - final validation
final_tests = [
    ("Tell me a short joke", "LocalLLMAgent"),
    ("Analyze this review: This product is terrible", "SentimentAnalysisAgent"),
    ("?", "CloudFallbackAgent")
]

for message, expected_agent in final_tests:
    result = orchestrator_agent(message)
    assert result.agent_name == expected_agent, f"FAIL: '{message}' => got {result.agent_name}, expected {expected_agent}"
    print(f"PASS: {message} => {result.agent_name}")

print()
print("✅ All final tests passed!")


PASS: Tell me a short joke => LocalLLMAgent
PASS: Analyze this review: This product is terrible => SentimentAnalysisAgent
PASS: ? => CloudFallbackAgent

✅ All final tests passed!


## סיכום

### מה עבד היטב
- **סוכן הניתוב** — הסיווג על בסיס מילות מפתח עבד בצורה אמינה ויציבה, ואפשר לנו לנתב כל בקשה לסוכן הנכון.
- **סוכן הסנטימנט** — הגישה מבוססת הכללים זיהתה נכון ביקורות חיוביות ושליליות. כאשר חבילת `transformers` מותקנת, ניתן לשדרג בקלות למודל מקצועי.
- **מנגנון הגיבוי** — המערכת טיפלה בצורה חלקה במקרים שבהם רמת הביטחון הייתה נמוכה, ועברה אוטומטית לסוכן הגיבוי.
- **סוכן הביקורת** — שילוב ה-Critic Agent בתוך סוכן התזמור מבטיח שתשובות באיכות נמוכה (ציון < 3) מועברות אוטומטית לטיפול חלופי.
- **הזיכרון** — שמירת ההקשר אפשרה טיפול בבקשות המשך כמו "למה?" בהתבסס על התוצאה הקודמת.

### מה היה מוגבל
- **סיווג דטרמיניסטי** — המימוש מבוסס מילות מפתח, ולכן לא מזהה ניואנסים או כוונות מורכבות. במערכת ייצור, היינו משתמשים במודל שפה לסיווג.
- **זיכרון חד‑פעמי** — הזיכרון שומר רק את הפנייה האחרונה, ולא היסטוריה מלאה של השיחה.
- **סנטימנט מבוסס כללים** — ללא חבילת `transformers`, הניתוח מוגבל לרשימת מילים קבועה ולא מבין הקשר.

### מודל מקומי לעומת שירות ענן
| קריטריון | מודל מקומי / API חסכוני (gpt-4o-mini) | שירות ענן מלא (gpt-4o, Claude וכו׳) |
|---|---|---|
| **עלות** | נמוכה מאוד | גבוהה יחסית |
| **מהירות** | מהיר — תגובות בשניות בודדות | תלוי בעומס, לרוב מהיר |
| **איכות** | טובה למשימות פשוטות ובינוניות | מצוינת למשימות מורכבות |
| **פרטיות** | הנתונים עוברים ל‑API חיצוני | גם כאן הנתונים עוברים לענן |
| **מתי לבחור?** | שיחה רגילה, סיכום קצר, סיווג בסיסי | ניתוח מורכב, יצירת תוכן מעמיק, נושאים רגישים |

> **במימוש זה** השתמשנו ב‑OpenAI API עם מודל `gpt-4o-mini` כתחליף ל‑Ollama. הגישה הזו מאפשרת לנו ליהנות מאיכות גבוהה בעלות נמוכה, תוך שמירה על מנגנון fallback מקומי למקרה שה‑API לא זמין.